# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset name and description
print("Dataset Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities (record sets, fields, columns, etc.) are referenced by their `@id` as per the Croissant specification.

In [ ]:
# List all record sets and their fields by @id

record_sets = dataset.metadata.record_sets
print(f"Number of record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    print("  Fields:")
    for field in rs['field']:
        # Each field is a dict with at least @id and name
        print(f"    Field @id: {field['@id']} | name: {field.get('name', 'N/A')}")
    print('-' * 50)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (using their @id)
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  [Warning] No records extracted for {record_set_id}.")
    else:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Extracted {len(records)} records, columns: {dataframes[record_set_id].columns.tolist()}")
    print()
# Display the first few rows of the main record set (choose the first one)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    if main_record_set_id in dataframes:
        print(f"Showing head of DataFrame for {main_record_set_id}:")
        display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

All `@id`s should be used to reference fields (columns) for clear correspondence with the schema.

In [ ]:
# Example EDA: Filtering by a numeric field and normalizing

from IPython.display import display
import numpy as np

# Pick a record set and choose numeric and grouping fields by their `@id`

# NOTE: You may adjust these based on the actual @id values found in section 2.
record_set_id = main_record_set_id  # Use the first record set found for demonstration
df = dataframes[record_set_id]

print(f"Columns in {record_set_id}: {df.columns.tolist()}")

# Find likely numeric fields by dtype or name (here we pick the first float/int column as an example)
numeric_field_id = None
group_field_id = None

for col in df.columns:
    if np.issubdtype(df[col].dtype, np.number) and numeric_field_id is None:
        numeric_field_id = col
    if df[col].dtype == object and group_field_id is None:
        group_field_id = col

print(f"Numeric field chosen: {numeric_field_id}")
print(f"Grouping field chosen: {group_field_id}\n")

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst rows with normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field, if present
    if group_field_id is not None:
        group_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        display(group_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
if numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot by group
if group_field_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, examine, and analyze a FAIR dataset defined by a Croissant schema. We reviewed record sets and field identifiers by `@id`, loaded records into pandas DataFrames, performed basic EDA by filtering and normalizing a numeric field, and visualized data distributions. For in-depth analysis, adjust the field `@id`s to match your specific use-case, and expand the analysis with domain-specific questions.